# recs_015_002 — D4 fine-tuned CE + hybrid (focused follow-up)

Follow-up to [`recs_015_ranker_d4_cross_encoder.ipynb`](recs_015_ranker_d4_cross_encoder.ipynb). **One question:** after fine-tuning the cross-encoder on Steam, does **FT CE + retr + pop** beat **D1** on val?

## Why a second notebook?

recs_015 ran many experiments at once. The important gap:

| | recs_015 | recs_015_002 (this notebook) |
| --- | --- | --- |
| Train CE on Steam | Phase B ✓ | **Reuse** saved model (or train if missing) |
| Hybrid `(w, α)` tuning | Used **zero-shot** CE scores | Uses **fine-tuned** CE scores |
| Val methods | 7+ variants (ZS CE, ZS hybrids, FT-only, …) | **3 contenders** vs D1 |

```mermaid
flowchart TB
  subgraph recs015["recs_015 (exploration)"]
    ZS["Zero-shot CE scores"]
    ZS --> PA["Phase A: tune w, α"]
    PA --> HY_ZS["Hybrid on VAL\n(ce + retr + pop)"]
    FTtrain["Phase B: fine-tune CE"]
    FTtrain --> FTonly["FT CE-only on VAL"]
  end
  subgraph recs002["recs_015_002 (this notebook)"]
    FTmodel["Load FT CE model"]
    FTmodel --> FTscores["Precompute FT CE scores"]
    FTscores --> PA2["Tune w, α on train_tune"]
    PA2 --> HY_FT["Hybrid on VAL\n(same formula, FT scores)"]
    FTscores --> FTonly2["FT CE-only on VAL"]
  end
  D1["D1 logpop baseline"] --> GATE{"Beat D1 NDCG@10\noverall + slice A?"}
  HY_FT --> GATE
  FTonly2 --> GATE
  GATE -->|yes| PROMOTE["Consider promoting"]
  GATE -->|no| KILL["Kill D4; ship D1"]
```

## The two D4 recipes (same FT model, different ranking)

Both use **one** fine-tuned cross-encoder. Neither retrains CE in this notebook unless the saved model is missing.

| Method | Formula | Tuning |
| --- | --- | --- |
| `two_tower_v1_ranker_d4_cross_encoder_ft_v1` | `norm(ce)` only | CE weights from Phase B (recs_015) |
| `two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1` | `w·norm(ce) + (1−w)·(α·norm(retr) + (1−α)·norm(log_pop))` | **`w`, `α` grid on train_tune** (this notebook) |

- **`w`** — CE weight vs retr+pop blend (grid `{0.1,…,1.0}`, CE required)
- **`α`** — retrieval vs popularity inside the `(1−w)` term (grid `{0.0,…,1.0}`)

**Promotion bar:** beat `two_tower_v1_heuristic_logpop_blend` (D1) on external val **NDCG@10** overall + slice A. Personalization reported for context only.

**Prereqs:** completed recs_015 Phase B (saved model) or set `TRAIN_CE_FT=True` below. Same pool/cohort jobs as recs_015.

```bash
pip install -e '.[cross-encoder]'
# FT model expected at: artifacts/recs/rankers/d4_cross_encoder_ft_v1/cross_encoder_model
```

## Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import (
    cohort_parquet_path,
    load_retrieval_pool_rows,
    load_retrieval_pools_jsonl,
)
from steam_review_ml.evaluation.heuristic_ranker import (
    METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND,
    pool_rerank_registry,
    rerank_scores_on_pool,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    _table_personalization,
    average_precision_at_k,
    hit_rate_at_k,
    load_eval_examples_from_parquet,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
)
from steam_review_ml.recommender.retrieve import ContentRetriever
from steam_review_ml.recommender.ranker_d4_cross_encoder import (
    METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_LOGPOP_BLEND_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_FT_V1,
    CrossEncoderConfig,
    CrossEncoderFinetuneConfig,
    build_app_candidate_texts,
    load_cross_encoder,
    make_ce_cache_score_fn,
    make_ce_retr_logpop_blend_score_fn,
    precompute_ce_scores_by_ex_idx,
    query_text_lookup_from_cohort_parquet,
    query_text_map_from_examples,
    query_text_map_from_train_pools,
    train_cross_encoder_ranker,
    tune_ce_retr_logpop_blend,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

POOL_METHOD = "two_tower_v1"
K_FINAL = 10
K_PERSONALIZATION = 10
MIN_REVIEW_CHARS = 30
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPLIT_SEED = 2027
TUNE_FRAC = 0.10
MAX_VAL_EXAMPLES: int | None = None
TRAIN_CE_FT = True  # True only if FT checkpoint missing (slow)

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
TRAIN_COHORT_PARQUET = cohort_parquet_path(REPO_ROOT / "artifacts/recs/eval_cache/train_ranker_v1")
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_COHORT_PARQUET = cohort_parquet_path(REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1")
GAME_PROFILE_REVIEWS = REPO_ROOT / "artifacts/recs/embeddings/game_profile/default/game_profile_reviews.parquet"
D4_FT_OUTPUT_DIR = REPO_ROOT / "artifacts/recs/rankers/d4_cross_encoder_ft_v1"
TUNE_PARAMS_PATH = D4_FT_OUTPUT_DIR / "ft_logpop_blend_tune_params.json"

CE_CFG = CrossEncoderConfig()
CE_FT_CFG = CrossEncoderFinetuneConfig(
    epochs=7,
    early_stopping_patience=2,
    max_train_pairs=1_000_000,   # not 3M+
    batch_size=8,              # default 16 — helps GPU RAM a little
    query_max_chars=256,       # default 512
    doc_max_chars=256,
)

for p in (TRAIN_POOLS_PARQUET, TRAIN_COHORT_PARQUET, VAL_JSONL, VAL_COHORT_PARQUET, GAME_PROFILE_REVIEWS):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

print(f"FT model dir: {D4_FT_OUTPUT_DIR / 'cross_encoder_model'}")
print(f"TRAIN_CE_FT={TRAIN_CE_FT}  MAX_VAL_EXAMPLES={MAX_VAL_EXAMPLES}")

FT model dir: /home/ryanr/workspace/steam_recommendations/artifacts/recs/rankers/d4_cross_encoder_ft_v1/cross_encoder_model
TRAIN_CE_FT=True  MAX_VAL_EXAMPLES=None


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data + train_tune split

In [2]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)
if MAX_VAL_EXAMPLES is not None:
    val_pools = val_pools[: int(MAX_VAL_EXAMPLES)]

val_examples = load_eval_examples_from_parquet(VAL_COHORT_PARQUET)
train_query_lookup = query_text_lookup_from_cohort_parquet(TRAIN_COHORT_PARQUET)
train_query_by_ex_idx = query_text_map_from_train_pools(train_pools, examples_by_key=train_query_lookup)
val_query_by_ex_idx = query_text_map_from_examples(val_examples)

candidate_texts = build_app_candidate_texts(
    GAME_PROFILE_REVIEWS,
    max_chars_per_app=CE_CFG.candidate_text_max_chars,
)

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row


def stratified_ex_idx_split(
    pools: list[dict[str, Any]], *, tune_frac: float, seed: int
) -> tuple[set[int], set[int]]:
    rng = np.random.default_rng(seed)
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    fit_ids: set[int] = set()
    tune_ids: set[int] = set()
    for ids in by_slice.values():
        ids_arr = np.asarray(sorted(ids))
        rng.shuffle(ids_arr)
        n_tune = max(1, int(round(len(ids_arr) * tune_frac)))
        tune_ids.update(int(x) for x in ids_arr[:n_tune])
        fit_ids.update(int(x) for x in ids_arr[n_tune:])
    return fit_ids, tune_ids


fit_ex_idx, tune_ex_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_fit = [r for r in train_pools if int(r["ex_idx"]) in fit_ex_idx]
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_ex_idx]

print(
    f"train fit={len(train_fit):,}  tune={len(train_tune):,}  "
    f"val={len(val_pools):,}  catalog={len(app_ids):,}"
)

train fit=46,522  tune=5,169  val=12,500  catalog=315


## Step 1 — Load fine-tuned CE (or train if missing)

Expect checkpoint from recs_015 Phase B. Set `TRAIN_CE_FT=True` to train here (slow).

In [ ]:
saved_ft = D4_FT_OUTPUT_DIR / "cross_encoder_model"

if saved_ft.is_dir():
    print(f"Loading FT CE from {saved_ft}")
    ce_model_ft = load_cross_encoder(model_path=saved_ft)
elif TRAIN_CE_FT:
    print("Training FT CE (recs_015 Phase B equivalent)...")
    ce_model_ft, _ft_history = train_cross_encoder_ranker(
        fit_pools=train_fit,
        tune_pools=train_tune,
        query_text_by_ex_idx=train_query_by_ex_idx,
        candidate_texts=candidate_texts,
        app_ids=app_ids,
        app_to_row=app_to_row,
        output_dir=D4_FT_OUTPUT_DIR,
        cfg=CE_FT_CFG,
        score_cfg=CE_CFG,
        k_final=K_FINAL,
    )
else:
    raise FileNotFoundError(
        f"Missing {saved_ft}. Run recs_015 Phase B or set TRAIN_CE_FT=True."
    )

Training FT CE (recs_015 Phase B equivalent)...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2993.35it/s]


Step,Training Loss
500,0.114123
1000,0.106723
1500,0.099397
2000,0.104316
2500,0.098839
3000,0.112509
3500,0.124089
4000,0.102301
4500,0.104626
5000,0.107857


D4-FT epoch 1/7: tune_NDCG@10=0.1465


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  6.06it/s]


Step,Training Loss
500,0.107082
1000,0.103165
1500,0.111074
2000,0.111244
2500,0.100108
3000,0.103451
3500,0.103336
4000,0.106646
4500,0.094742
5000,0.101765


## Step 2 — Precompute **fine-tuned** CE scores

Run once on train_tune (for blending) and val (for reporting). ~100 forward passes per example.

In [2]:
print("Precomputing FT CE on train_tune...")
ce_tune_ft = precompute_ce_scores_by_ex_idx(
    ce_model_ft,
    train_tune,
    query_text_by_ex_idx=train_query_by_ex_idx,
    candidate_texts=candidate_texts,
    cfg=CE_CFG,
)

print("Precomputing FT CE on val...")
ce_val_ft = precompute_ce_scores_by_ex_idx(
    ce_model_ft,
    val_pools,
    query_text_by_ex_idx=val_query_by_ex_idx,
    candidate_texts=candidate_texts,
    cfg=CE_CFG,
)
print(f"cached tune={len(ce_tune_ft):,}  val={len(ce_val_ft):,} pools")

Precomputing FT CE on train_tune...


NameError: name 'precompute_ce_scores_by_ex_idx' is not defined

## Step 3 — Tune `(w, α)` on train_tune using **FT CE** scores

This is what recs_015 Phase A did **not** do (015 used zero-shot CE for this grid).

In [ ]:
best_w, best_alpha, tune_ndcg = tune_ce_retr_logpop_blend(
    train_tune,
    ce_tune_ft,
    pop_row=pop_row,
    app_to_row=app_to_row,
    app_ids=app_ids,
    k_final=K_FINAL,
)
print(
    f"FT hybrid: best_w={best_w:.1f}  best_alpha={best_alpha:.1f}  "
    f"train_tune NDCG@{K_FINAL}={tune_ndcg:.4f}"
)

tune_params = {
    "ce_source": "fine_tuned",
    "best_w": best_w,
    "best_alpha": best_alpha,
    "train_tune_ndcg_at_k": tune_ndcg,
    "k_final": K_FINAL,
}
TUNE_PARAMS_PATH.parent.mkdir(parents=True, exist_ok=True)
TUNE_PARAMS_PATH.write_text(json.dumps(tune_params, indent=2) + "\n", encoding="utf-8")
print(f"Saved tune params → {TUNE_PARAMS_PATH}")

## Step 4 — Val head-to-head vs D1

Four methods that matter for the promote/kill decision (+ optional context rows).

In [ ]:
def pool_scores_to_ranked_indices(
    pool_app_ids: list[int],
    pool_scores: np.ndarray,
    *,
    k_final: int,
) -> np.ndarray:
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def popularity_catalog_scores(*, query_app_id: int) -> np.ndarray:
    s = np.asarray(pop_row, dtype=np.float64).copy()
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def popularity_catalog_ranked_indices(*, query_app_id: int, k_final: int) -> np.ndarray:
    return _rank_rows(popularity_catalog_scores(query_app_id=query_app_id))[:k_final]


def full_catalog_scores_for_pool_row(
    row: dict[str, Any],
    *,
    score_fn: Callable[..., np.ndarray] | None = None,
    catalog_pop: bool = False,
) -> np.ndarray:
    if catalog_pop:
        return popularity_catalog_scores(query_app_id=int(row["query_app_id"]))
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    if score_fn is None:
        pool_scores = np.asarray(ret_sc, dtype=np.float64)
    else:
        pool_scores = np.asarray(
            score_fn(pool_apps, ret_sc, ex_idx=int(row["ex_idx"])), dtype=np.float64
        )
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_apps, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return full


def per_example_metrics(
    row: dict[str, Any],
    *,
    method: str,
    score_fn: Callable[..., np.ndarray] | None = None,
    catalog_pop: bool = False,
) -> dict[str, Any] | None:
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        return None
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    if catalog_pop:
        ranked = popularity_catalog_ranked_indices(
            query_app_id=int(row["query_app_id"]), k_final=K_FINAL
        )
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(ret_sc), k_final=K_FINAL)
    else:
        blend = score_fn(pool_apps, ret_sc, ex_idx=int(row["ex_idx"]))
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
    return {
        "method": method,
        "slice_name": row.get("slice_name", ""),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
    }


D1_SPEC = pool_rerank_registry()[METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND]


def score_d1_logpop(pool_apps, retrieval_scores, **_ignored):
    return rerank_scores_on_pool(
        pool_apps, retrieval_scores, D1_SPEC, pop_row=pop_row, app_to_row=app_to_row
    )


HEAD_TO_HEAD: list[dict[str, Any]] = [
    {"method": POOL_METHOD, "kind": "pool_retrieval"},
    {"method": METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND, "kind": "pool_rerank", "score_fn": score_d1_logpop},
    {
        "method": METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_FT_V1,
        "kind": "pool_rerank",
        "score_fn": make_ce_cache_score_fn(ce_val_ft),
    },
    {
        "method": METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_LOGPOP_BLEND_V1,
        "kind": "pool_rerank",
        "score_fn": make_ce_retr_logpop_blend_score_fn(
            ce_val_ft,
            pop_row=pop_row,
            app_to_row=app_to_row,
            w=best_w,
            alpha=best_alpha,
        ),
    },
]

val_rows: list[dict[str, Any]] = []
for i, row in enumerate(val_pools):
    if i and i % 500 == 0:
        print(f"scored {i:,}/{len(val_pools):,}...", flush=True)
    for spec in HEAD_TO_HEAD:
        if spec["kind"] == "pool_retrieval":
            m = per_example_metrics(row, method=spec["method"], score_fn=None)
        else:
            m = per_example_metrics(row, method=spec["method"], score_fn=spec["score_fn"])
        if m is not None:
            val_rows.append(m)

df_val = pd.DataFrame(val_rows)
print(f"methods: {sorted(df_val['method'].unique())}")

## Personalization (guardrails)

In [ ]:
for i, ex in enumerate(val_examples):
    ex["ex_idx"] = i

pools_by_ex = {int(r["ex_idx"]): r for r in val_pools}
pool_ex_indices = set(pools_by_ex.keys())


def _make_pool_catalog_scorer(
    score_fn: Callable[..., np.ndarray] | None = None,
) -> Callable[[dict[str, Any]], np.ndarray]:
    def scorer(ex: dict[str, Any]) -> np.ndarray:
        return full_catalog_scores_for_pool_row(
            pools_by_ex[int(ex["ex_idx"])], score_fn=score_fn
        )

    return scorer


person_methods: dict[str, Callable[[dict[str, Any]], np.ndarray]] = {
    "popularity_train": lambda ex: popularity_catalog_scores(query_app_id=int(ex["query_app_id"])),
}
for spec in HEAD_TO_HEAD:
    if spec["kind"] == "pool_retrieval":
        person_methods[spec["method"]] = _make_pool_catalog_scorer(score_fn=None)
    else:
        person_methods[spec["method"]] = _make_pool_catalog_scorer(score_fn=spec["score_fn"])

retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
df_person = _table_personalization(
    methods=person_methods,
    examples=val_examples,
    X=retriever.embedding_matrix,
    app_ids=app_ids,
    pop_row=pop_row,
    k_personalization=K_PERSONALIZATION,
    example_indices=pool_ex_indices,
)

person_cols = [
    f"ILD@{K_PERSONALIZATION}",
    f"CatalogCoverage@{K_PERSONALIZATION}",
    f"Novelty@{K_PERSONALIZATION}",
    f"PersonalizationGapVsPopularity@{K_PERSONALIZATION}",
]
df_val_overall = df_val.groupby("method")[["Hit@K", "MAP@K", "NDCG@K", "MRR"]].mean().reset_index()
df_val_full = df_val_overall.merge(df_person, on="method", how="left")
gap_col = f"PersonalizationGapVsPopularity@{K_PERSONALIZATION}"

## Results + promote / kill

In [ ]:
display(Markdown("### Val relevance + personalization (overall)"))
display(
    df_val_full.sort_values("NDCG@K", ascending=False)[
        ["method", "Hit@K", "NDCG@K", "MRR", *person_cols]
    ]
)

display(Markdown("### By slice (NDCG@K)"))
display(
    df_val.groupby(["slice_name", "method"])[["Hit@K", "NDCG@K", "MRR"]]
    .mean()
    .sort_values(["slice_name", "NDCG@K"], ascending=[True, False])
)

d1_ndcg = float(
    df_val.loc[df_val["method"] == METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND, "NDCG@K"].mean()
)
d4_candidates = [
    METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_LOGPOP_BLEND_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_FT_V1,
]
print(f"\nD1 val NDCG@{K_FINAL}={d1_ndcg:.4f}  (promotion bar)")
print(f"FT hybrid tuned: w={best_w:.1f}  alpha={best_alpha:.1f}")
for m in d4_candidates:
    if m not in df_val["method"].values:
        continue
    nd = float(df_val.loc[df_val["method"] == m, "NDCG@K"].mean())
    verdict = "BEATS D1" if nd > d1_ndcg else "below D1"
    print(f"  {m}: NDCG@{K_FINAL}={nd:.4f} ({verdict})")

slice_a = df_val.loc[df_val["slice_name"] == "A"]
if not slice_a.empty:
    d1_a = float(slice_a.loc[slice_a["method"] == METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND, "NDCG@K"].mean())
    print(f"\nSlice A: D1 NDCG@{K_FINAL}={d1_a:.4f}")
    for m in d4_candidates:
        sub = slice_a.loc[slice_a["method"] == m, "NDCG@K"]
        if sub.empty:
            continue
        nd_a = float(sub.mean())
        verdict = "BEATS D1" if nd_a > d1_a else "below D1"
        print(f"  {m}: NDCG@{K_FINAL}={nd_a:.4f} ({verdict})")

### Takeaway

- **recs_015** explored many CE variants; the hybrid used **zero-shot** CE for `(w, α)` tuning.
- **This notebook** is the missing experiment: **same hybrid formula, fine-tuned CE scores**.
- **Promote** only if FT hybrid beats D1 on val NDCG@10 **overall and slice A**; otherwise **kill D4** and ship D1.
- Tune params saved to `artifacts/recs/rankers/d4_cross_encoder_ft_v1/ft_logpop_blend_tune_params.json`.